<img src="_static/pism_logo_transp.png" alt="pism-terra" width="240" align="left"/>

# PISM-Cloud — interactive app

Run this notebook with **Voila** to get a button-driven web UI (no need to run cells by hand):

```
voila notebooks/pism_cloud_app.ipynb --strip_sources=True
```

It wraps the same workflow as `pism_cloud.ipynb`: connect with your Earthdata (EDL) login,
upload a PISM config / template / (optional) UQ file, submit a *prepare* job, watch it,
then find the generated run scripts and *execute* them. AWS credentials come from your
local environment (the default boto3 chain / `~/.aws`).

In [ ]:
from copy import deepcopy

import boto3
import s3fs
import hyp3_sdk as sdk
import ipywidgets as widgets
from IPython.display import display

# Single shared session state (single kernel / single user).
state = {
    "client": None,
    "jobs": None,
    "pism_config": None,
    "pism_template": None,
    "pism_uq": "none",
}

_s3 = boto3.client("s3")

## 1. Connect to PISM-Cloud

Enter your [Earthdata Login](https://urs.earthdata.nasa.gov/) credentials and click **Connect**.

In [ ]:
test_cloud_server = "https://pism-cloud-test.asf.alaska.edu"
production_cloud_server = "https://pism-cloud.asf.alaska.edu"

# Drop down menu to choose between the PISM Cloud Production and PISM Cloud Test
# servers. The default is the production Cloud.
server_w = widgets.Dropdown(
    options=[
        ("PISM Cloud Production", production_cloud_server),
        ("PISM Cloud Test", test_cloud_server),
    ],
    value=production_cloud_server,
    description="Server:",
    layout=widgets.Layout(width="520px"),
)
user_w = widgets.Text(description="EDL user:")
pass_w = widgets.Password(description="EDL pass:")
connect_btn = widgets.Button(description="Connect", button_style="primary", icon="plug")
connect_out = widgets.Output()


def _connect(_):
    connect_out.clear_output()
    with connect_out:
        try:
            client = sdk.HyP3(server_w.value, username=user_w.value, password=pass_w.value)
            state["client"] = client
            print(f"\u2705 Connected. Credits: {client.check_credits()}")
        except Exception as exc:  # noqa: BLE001
            print(f"\u274c Connection failed: {exc}")


connect_btn.on_click(_connect)
display(widgets.VBox([server_w, user_w, pass_w, connect_btn, connect_out]))

## 2. Glacier & metadata

In [ ]:
rgi_w = widgets.Text(value="RGI2000-v7.0-C-01-04374", description="RGI ID:",
                     layout=widgets.Layout(width="420px"))
name_w = widgets.Text(value="summer_school", description="Name:")
project_w = widgets.Text(value="test_inverse_fridays", description="Project:")
bucket_w = widgets.Text(value="pism-cloud-data", description="Bucket:")
ntasks_w = widgets.IntText(value=32, description="ntasks:")
display(widgets.VBox([rgi_w, name_w, project_w, bucket_w, ntasks_w]))

## 3. Upload input files

Upload the PISM `config` (`.toml`) and `template` (`.j2`); the `UQ` file (`.toml`) is optional.
Each file is shipped to `s3://{bucket}/glacier/{name}/{kind}/`.

In [ ]:
def _make_uploader(kind: str, state_key: str, accept: str = "") -> widgets.VBox:
    header = widgets.HTML(value=f"<b>Upload {kind} file</b>")
    picker = widgets.FileUpload(accept=accept, multiple=False, description=f"Browse {kind}…")
    status = widgets.HTML(value="<i>no file uploaded yet</i>")

    def _on_change(_change):
        if not picker.value:
            return
        # ipywidgets >= 8 returns a tuple of dicts; older versions return a dict.
        item = picker.value[0] if isinstance(picker.value, (list, tuple)) else next(iter(picker.value.values()))
        filename = item.get("name") or item["metadata"]["name"]
        raw = item["content"] if isinstance(item.get("content"), bytes) else bytes(item["content"])
        key = f"glacier/{name_w.value}/{kind}/{filename}"
        try:
            _s3.put_object(Bucket=bucket_w.value, Key=key, Body=raw)
        except Exception as exc:  # noqa: BLE001
            status.value = f"❌ upload failed: {exc}"
            return
        uri = f"s3://{bucket_w.value}/{key}"
        state[state_key] = uri
        status.value = f"✅ uploaded → <code>{uri}</code>"

    picker.observe(_on_change, names="value")
    return widgets.VBox([header, picker, status])


display(
    widgets.VBox(
        [
            _make_uploader("config", "pism_config", accept=".toml"),
            _make_uploader("template", "pism_template", accept=".j2,.jinja,.jinja2"),
            _make_uploader("uq", "pism_uq", accept=".toml"),
        ]
    )
)

## 4. Run type & submit the prepare job

In [ ]:
runtype_w = widgets.RadioButtons(options=["forward", "inverse"], description="Run type:")
submit_btn = widgets.Button(description="Submit prepare job", button_style="primary", icon="rocket")
submit_out = widgets.Output()

_JOB_TYPE = {"forward": "PISM_TERRA_RUN_FORWARD", "inverse": "PISM_TERRA_RUN_INVERSE"}


def _submit_prepare(_):
    submit_out.clear_output()
    with submit_out:
        if state["client"] is None:
            print("❌ Not connected — click Connect in step 1 first.")
            return
        if not state["pism_config"] or not state["pism_template"]:
            print("❌ Upload a config (.toml) and a template (.j2) first.")
            return
        # Literal {name}/{job_id} are filled in by the PISM-Cloud backend.
        bucket_prefix = f"{{name}}/{project_w.value}/{{job_id}}"
        pism_job = {
            "job_type": _JOB_TYPE[runtype_w.value],
            "name": name_w.value,
            "bucket": bucket_w.value,
            "bucket_prefix": bucket_prefix,
            "job_parameters": {
                "rgi_id": rgi_w.value,
                "pism_config": state["pism_config"],
                "run_template": state["pism_template"],
                "uq_config": state["pism_uq"],
                "ntasks": ntasks_w.value,
            },
        }
        try:
            jobs = state["client"].submit_prepared_jobs(pism_job)
        except Exception as exc:  # noqa: BLE001
            print(f"❌ Submit failed: {exc}")
            return
        state["jobs"] = jobs
        print(f"✅ Submitted {len(jobs)} job(s):")
        for j in jobs:
            print(f"  {j.job_id}  {j.job_type}  {j.status_code}")


submit_btn.on_click(_submit_prepare)
display(widgets.VBox([runtype_w, submit_btn, submit_out]))

## 5. Job status

Click **Refresh status** to poll once (non-blocking), or tick auto-refresh to poll every 30 s.

In [ ]:
import threading

refresh_btn = widgets.Button(description="Refresh status", icon="refresh")
autorefresh_w = widgets.Checkbox(value=False, description="Auto-refresh (30s)")
status_out = widgets.Output()
_timer = {"t": None}


def _render_status():
    status_out.clear_output()
    with status_out:
        jobs = state["jobs"]
        if not jobs:
            print("No jobs submitted yet.")
            return
        try:
            jobs = state["client"].refresh(jobs)
        except Exception as exc:  # noqa: BLE001
            print(f"❌ Refresh failed: {exc}")
            return
        state["jobs"] = jobs
        for j in jobs:
            print(f"  {j.job_id}  {j.job_type}  {j.status_code}")


def _refresh(_):
    _render_status()


def _schedule():
    if not autorefresh_w.value:
        return
    _render_status()
    timer = threading.Timer(30.0, _schedule)
    timer.daemon = True
    _timer["t"] = timer
    timer.start()


def _toggle_auto(change):
    if change["new"]:
        _schedule()
    elif _timer["t"] is not None:
        _timer["t"].cancel()


refresh_btn.on_click(_refresh)
autorefresh_w.observe(_toggle_auto, names="value")
display(widgets.VBox([widgets.HBox([refresh_btn, autorefresh_w]), status_out]))

## 6. Execute the simulations

Once the prepare job(s) have **succeeded**, this finds the generated run scripts in S3 and
submits one `PISM_TERRA_EXECUTE` job per script. Track them with **Refresh status** above.

In [ ]:
def get_run_scripts(job) -> list[str]:
    fs = s3fs.S3FileSystem(anon=True)
    files = fs.ls(f"{bucket_w.value}/{job.name}/{project_w.value}/{job.job_id}/{rgi_w.value}/run_scripts/")
    return [f"s3://{file}" for file in files]


execute_btn = widgets.Button(description="Find scripts & execute", button_style="primary", icon="play")
execute_out = widgets.Output()


def _execute(_):
    execute_out.clear_output()
    with execute_out:
        if state["client"] is None:
            print("❌ Not connected.")
            return
        jobs = state["jobs"]
        if not jobs:
            print("❌ No prepare jobs found — submit them in step 4 and let them finish first.")
            return
        execute_template = {
            "name": name_w.value,
            "bucket": bucket_w.value,
            "bucket_prefix": f"{{name}}/{project_w.value}/{{job_id}}",
            "job_type": "PISM_TERRA_EXECUTE",
            "job_parameters": {},
        }
        prepared = []
        for job in jobs:
            try:
                scripts = get_run_scripts(job)
            except Exception as exc:  # noqa: BLE001
                print(f"⚠️  could not list run scripts for {job.job_id}: {exc}")
                continue
            for script in scripts:
                job_dict = deepcopy(execute_template)
                job_dict["name"] = job.name
                job_dict["job_parameters"]["run_script"] = script
                prepared.append(job_dict)
        if not prepared:
            print("No run scripts found yet (prepare jobs may still be running).")
            return
        try:
            exec_jobs = state["client"].submit_prepared_jobs(prepared)
        except Exception as exc:  # noqa: BLE001
            print(f"❌ Execute submit failed: {exc}")
            return
        state["jobs"] = exec_jobs
        print(f"✅ Submitted {len(exec_jobs)} execute job(s). Track them with Refresh status above.")
        for j in exec_jobs:
            print(f"  {j.job_id}  {j.status_code}")


execute_btn.on_click(_execute)
display(widgets.VBox([execute_btn, execute_out]))

## Analysis

Go to [Open Science Lab](https://opensciencelab.asf.alaska.edu/) to analyze the results, or download them.